In [21]:
import datetime
import pandas as pd
from netCDF4 import Dataset
import numpy as np
import os
import xarray as xr

In [22]:
datapath = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2SMOS/'

datapath_aux = '/Users/carmennab/Dropbox/pb_new/auxiliary/'
ease_lons = np.load(f'{datapath_aux}lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}lat_cent.npy')

In [23]:
years = np.arange(2010,2023)

for year in years:
    d1 = datetime.date(year, 1, 1)
    d2 = datetime.date(year, 12, 31)
    dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]
    str_dates = [date.strftime('%Y%m%d') for date in dates]
    
    yearly_sits = np.full((len(dates),ease_lons.shape[0],ease_lons.shape[1]),np.nan)
    yearly_snows = np.full((len(dates),ease_lons.shape[0],ease_lons.shape[1]),np.nan)
    
    for count,str_date in enumerate(str_dates):
        filepath = f'{datapath}{year}/{str_date}.nc'
        
        if os.path.exists(filepath) == True: 
            file = Dataset(f'{datapath}{year}/{str_date}.nc')
            sit = file['sea_ice_thickness']
        
            yearly_sits[count] = np.array(sit)
        else:pass
        
    ### create NETCDF
    filepath_out = f'{datapath}yearly_files/{str_dates[0]}-{str_dates[-1]}.nc'
    data_vars = {'sea_ice_thickness':(['t','x','y'],yearly_sits)}
    ds = xr.Dataset(data_vars = data_vars,
                    coords={'date':(['t'],str_dates),
                            'longitude':(['x','y'],ease_lons),
                            'latitude':(['x','y'],ease_lats)} )
    ds.attrs = {'Description': "Daily-resolution sea ice thickness data on 25km EASE-Grid 2.0 (Nab et al. 2026)"}
    ds.to_netcdf(filepath_out,'w')